# Task B / Person B — CUDA Merge Sort

This notebook completes the CUDA part of the F.CSM306 assignment for **Merge Sort**.

It includes:

- CUDA GPU merge sort
- CPU sequential merge sort baseline
- tests for 10k, 100k, and 1M elements
- Host-to-Device transfer time
- CUDA kernel computation time
- Device-to-Host transfer time
- total execution time
- SpeedUp calculation
- CSV and graph output

Before running: enable a GPU runtime in Colab.


## 1. Check GPU and CUDA compiler

Run this first. If `nvidia-smi` fails, the notebook is not using a GPU runtime.


In [ ]:

!nvidia-smi
!nvcc --version


## 2. Write the CUDA source file

This cell creates the full CUDA C++ program.


In [ ]:
%%writefile task_b_cuda_merge_sort_colab.cu
/*
  F.CSM306 Parallel and Distributed Computing
  Task B / Person B - CUDA Merge Sort

  This program implements a GPU-based merge sort using CUDA.
  It measures:
    - Host to Device transfer time
    - CUDA kernel computation time
    - Device to Host transfer time
    - Total CUDA execution time
    - SpeedUp compared to a CPU sequential merge sort baseline

  Test sizes: 10k, 100k, 1M elements
*/

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <limits>
#include <random>
#include <string>
#include <vector>

#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                               \
    do {                                                                               \
        cudaError_t err = (call);                                                      \
        if (err != cudaSuccess) {                                                      \
            std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ << " -> "   \
                      << cudaGetErrorString(err) << std::endl;                         \
            std::exit(EXIT_FAILURE);                                                   \
        }                                                                              \
    } while (0)

struct Stats {
    double avg_ms = 0.0;
    double min_ms = 0.0;
    double max_ms = 0.0;
};

struct CudaTiming {
    double h2d_ms = 0.0;
    double kernel_ms = 0.0;
    double d2h_ms = 0.0;
    double total_ms = 0.0;
    bool correct = false;
};

// ------------------------- CPU Sequential Merge Sort -------------------------

void merge_range(std::vector<int>& values, std::vector<int>& temp, int left, int mid, int right) {
    int i = left;
    int j = mid;
    int k = left;

    while (i < mid && j < right) {
        if (values[i] <= values[j]) {
            temp[k++] = values[i++];
        } else {
            temp[k++] = values[j++];
        }
    }

    while (i < mid) {
        temp[k++] = values[i++];
    }

    while (j < right) {
        temp[k++] = values[j++];
    }

    for (int index = left; index < right; ++index) {
        values[index] = temp[index];
    }
}

void merge_sort_recursive(std::vector<int>& values, std::vector<int>& temp, int left, int right) {
    if (right - left <= 1) {
        return;
    }

    int mid = left + (right - left) / 2;
    merge_sort_recursive(values, temp, left, mid);
    merge_sort_recursive(values, temp, mid, right);
    merge_range(values, temp, left, mid, right);
}

void sequential_merge_sort(std::vector<int>& values) {
    std::vector<int> temp(values.size());
    merge_sort_recursive(values, temp, 0, static_cast<int>(values.size()));
}

// ------------------------- CUDA Parallel Merge Pass -------------------------

__device__ int lower_bound_device(const int* data, int start, int end, int value) {
    int low = start;
    int high = end;

    while (low < high) {
        int mid = low + (high - low) / 2;
        if (data[mid] < value) {
            low = mid + 1;
        } else {
            high = mid;
        }
    }

    return low - start;
}

__device__ int upper_bound_device(const int* data, int start, int end, int value) {
    int low = start;
    int high = end;

    while (low < high) {
        int mid = low + (high - low) / 2;
        if (data[mid] <= value) {
            low = mid + 1;
        } else {
            high = mid;
        }
    }

    return low - start;
}

/*
  One kernel pass merges sorted blocks of size width into sorted blocks of size 2*width.

  Instead of assigning one thread to one whole merge, this kernel assigns one CUDA thread
  to one element. Each thread calculates where its element belongs in the merged output
  by binary searching the opposite half.

  For duplicate values:
    - left half uses lower_bound
    - right half uses upper_bound
  This prevents two equal elements from writing to the same output position.
*/
__global__ void parallel_merge_pass_kernel(const int* input, int* output, int n, int width) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= n) {
        return;
    }

    int pair_width = width * 2;
    int pair_start = (idx / pair_width) * pair_width;

    int mid = pair_start + width;
    if (mid > n) {
        mid = n;
    }

    int pair_end = pair_start + pair_width;
    if (pair_end > n) {
        pair_end = n;
    }

    int value = input[idx];
    int output_position = idx;

    if (idx < mid) {
        int left_index = idx - pair_start;
        int elements_less_in_right = lower_bound_device(input, mid, pair_end, value);
        output_position = pair_start + left_index + elements_less_in_right;
    } else {
        int right_index = idx - mid;
        int elements_less_or_equal_in_left = upper_bound_device(input, pair_start, mid, value);
        output_position = pair_start + right_index + elements_less_or_equal_in_left;
    }

    output[output_position] = value;
}

// ------------------------- Timing Helpers -------------------------

Stats calculate_stats(const std::vector<double>& times) {
    Stats stats;
    if (times.empty()) {
        return stats;
    }

    double sum = 0.0;
    stats.min_ms = std::numeric_limits<double>::max();
    stats.max_ms = 0.0;

    for (double value : times) {
        sum += value;
        stats.min_ms = std::min(stats.min_ms, value);
        stats.max_ms = std::max(stats.max_ms, value);
    }

    stats.avg_ms = sum / static_cast<double>(times.size());
    return stats;
}

std::vector<int> generate_random_data(int n, int seed) {
    std::mt19937 rng(seed);
    std::uniform_int_distribution<int> dist(0, 1'000'000);

    std::vector<int> values(n);
    for (int& value : values) {
        value = dist(rng);
    }

    return values;
}

long long estimated_merge_sort_operations(int n) {
    int levels = static_cast<int>(std::ceil(std::log2(static_cast<double>(n))));
    return static_cast<long long>(n) * levels;
}

long long estimated_cuda_operations(int n) {
    int levels = static_cast<int>(std::ceil(std::log2(static_cast<double>(n))));

    // In this CUDA implementation, each merge pass uses a binary search to find
    // the output position of each element. This gives a simple estimate of work.
    return static_cast<long long>(n) * levels * levels;
}

double achievable_mops(long long operations, double time_ms) {
    if (time_ms <= 0.0) {
        return 0.0;
    }

    double seconds = time_ms / 1000.0;
    return static_cast<double>(operations) / seconds / 1'000'000.0;
}

CudaTiming run_cuda_merge_sort_once(const std::vector<int>& input,
                                    const std::vector<int>& reference,
                                    int block_size) {
    int n = static_cast<int>(input.size());
    size_t bytes = static_cast<size_t>(n) * sizeof(int);

    int* d_input = nullptr;
    int* d_temp = nullptr;

    CUDA_CHECK(cudaMalloc(&d_input, bytes));
    CUDA_CHECK(cudaMalloc(&d_temp, bytes));

    cudaEvent_t h2d_start, h2d_stop;
    cudaEvent_t kernel_start, kernel_stop;
    cudaEvent_t d2h_start, d2h_stop;

    CUDA_CHECK(cudaEventCreate(&h2d_start));
    CUDA_CHECK(cudaEventCreate(&h2d_stop));
    CUDA_CHECK(cudaEventCreate(&kernel_start));
    CUDA_CHECK(cudaEventCreate(&kernel_stop));
    CUDA_CHECK(cudaEventCreate(&d2h_start));
    CUDA_CHECK(cudaEventCreate(&d2h_stop));

    CudaTiming timing;
    float elapsed = 0.0f;

    // Host to Device transfer
    CUDA_CHECK(cudaEventRecord(h2d_start));
    CUDA_CHECK(cudaMemcpy(d_input, input.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(h2d_stop));
    CUDA_CHECK(cudaEventSynchronize(h2d_stop));
    CUDA_CHECK(cudaEventElapsedTime(&elapsed, h2d_start, h2d_stop));
    timing.h2d_ms = elapsed;

    // CUDA kernel computation
    int* current_input = d_input;
    int* current_output = d_temp;

    CUDA_CHECK(cudaEventRecord(kernel_start));
    for (int width = 1; width < n; width *= 2) {
        int blocks = (n + block_size - 1) / block_size;
        parallel_merge_pass_kernel<<<blocks, block_size>>>(current_input, current_output, n, width);
        CUDA_CHECK(cudaGetLastError());
        std::swap(current_input, current_output);
    }
    CUDA_CHECK(cudaEventRecord(kernel_stop));
    CUDA_CHECK(cudaEventSynchronize(kernel_stop));
    CUDA_CHECK(cudaEventElapsedTime(&elapsed, kernel_start, kernel_stop));
    timing.kernel_ms = elapsed;

    // Device to Host transfer
    std::vector<int> output(n);
    CUDA_CHECK(cudaEventRecord(d2h_start));
    CUDA_CHECK(cudaMemcpy(output.data(), current_input, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(d2h_stop));
    CUDA_CHECK(cudaEventSynchronize(d2h_stop));
    CUDA_CHECK(cudaEventElapsedTime(&elapsed, d2h_start, d2h_stop));
    timing.d2h_ms = elapsed;

    timing.total_ms = timing.h2d_ms + timing.kernel_ms + timing.d2h_ms;
    timing.correct = (output == reference) && std::is_sorted(output.begin(), output.end());

    CUDA_CHECK(cudaEventDestroy(h2d_start));
    CUDA_CHECK(cudaEventDestroy(h2d_stop));
    CUDA_CHECK(cudaEventDestroy(kernel_start));
    CUDA_CHECK(cudaEventDestroy(kernel_stop));
    CUDA_CHECK(cudaEventDestroy(d2h_start));
    CUDA_CHECK(cudaEventDestroy(d2h_stop));

    CUDA_CHECK(cudaFree(d_input));
    CUDA_CHECK(cudaFree(d_temp));

    return timing;
}

int main(int argc, char** argv) {
    int block_size = 256;
    int trials = 5;

    if (argc >= 2) {
        block_size = std::stoi(argv[1]);
    }

    if (argc >= 3) {
        trials = std::stoi(argv[2]);
    }

    if (block_size < 32 || block_size > 1024) {
        std::cerr << "Block size should normally be between 32 and 1024." << std::endl;
        return EXIT_FAILURE;
    }

    if (trials <= 0) {
        std::cerr << "Trial count must be greater than 0." << std::endl;
        return EXIT_FAILURE;
    }

    int device_count = 0;
    CUDA_CHECK(cudaGetDeviceCount(&device_count));
    if (device_count == 0) {
        std::cerr << "No CUDA-capable GPU found." << std::endl;
        return EXIT_FAILURE;
    }

    cudaDeviceProp prop{};
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));

    std::vector<int> sizes = {10'000, 100'000, 1'000'000};

    std::ofstream csv("task_b_cuda_results.csv");
    csv << "n,method,block_size,trials,avg_h2d_ms,avg_kernel_ms,avg_d2h_ms,avg_total_ms,"
        << "avg_cpu_ms,speedup_vs_cpu,data_transfer_bytes,estimated_operations,achievable_mops,correct\n";

    std::cout << "Task B - CUDA Merge Sort" << std::endl;
    std::cout << "GPU: " << prop.name << std::endl;
    std::cout << "Block size: " << block_size << ", trials: " << trials << "\n" << std::endl;

    std::cout << std::setw(10) << "n"
              << " | " << std::setw(14) << "CPU avg ms"
              << " | " << std::setw(14) << "CUDA total"
              << " | " << std::setw(12) << "kernel ms"
              << " | " << std::setw(8) << "speedup"
              << " | " << std::setw(8) << "correct" << std::endl;
    std::cout << std::string(82, '-') << std::endl;

    for (int n : sizes) {
        std::vector<int> input = generate_random_data(n, 12345 + n);

        std::vector<int> reference = input;
        std::sort(reference.begin(), reference.end());

        std::vector<double> cpu_times;
        cpu_times.reserve(trials);

        for (int trial = 0; trial < trials; ++trial) {
            std::vector<int> cpu_data = input;

            auto start = std::chrono::high_resolution_clock::now();
            sequential_merge_sort(cpu_data);
            auto end = std::chrono::high_resolution_clock::now();

            double ms = std::chrono::duration<double, std::milli>(end - start).count();
            cpu_times.push_back(ms);

            if (cpu_data != reference) {
                std::cerr << "CPU sequential result is incorrect for n=" << n << std::endl;
                return EXIT_FAILURE;
            }
        }

        Stats cpu_stats = calculate_stats(cpu_times);

        std::vector<double> h2d_times;
        std::vector<double> kernel_times;
        std::vector<double> d2h_times;
        std::vector<double> total_times;
        bool all_correct = true;

        for (int trial = 0; trial < trials; ++trial) {
            CudaTiming timing = run_cuda_merge_sort_once(input, reference, block_size);
            h2d_times.push_back(timing.h2d_ms);
            kernel_times.push_back(timing.kernel_ms);
            d2h_times.push_back(timing.d2h_ms);
            total_times.push_back(timing.total_ms);
            all_correct = all_correct && timing.correct;
        }

        Stats h2d_stats = calculate_stats(h2d_times);
        Stats kernel_stats = calculate_stats(kernel_times);
        Stats d2h_stats = calculate_stats(d2h_times);
        Stats total_stats = calculate_stats(total_times);

        size_t data_transfer_bytes = static_cast<size_t>(n) * sizeof(int) * 2;
        long long cpu_ops = estimated_merge_sort_operations(n);
        long long cuda_ops = estimated_cuda_operations(n);

        double speedup = cpu_stats.avg_ms / total_stats.avg_ms;

        csv << n << ",CPU Sequential,0," << trials << ",0,0,0,"
            << cpu_stats.avg_ms << "," << cpu_stats.avg_ms << ",1,0,"
            << cpu_ops << "," << achievable_mops(cpu_ops, cpu_stats.avg_ms) << ",yes\n";

        csv << n << ",CUDA GPU," << block_size << "," << trials << ","
            << h2d_stats.avg_ms << "," << kernel_stats.avg_ms << "," << d2h_stats.avg_ms << ","
            << total_stats.avg_ms << "," << cpu_stats.avg_ms << "," << speedup << ","
            << data_transfer_bytes << "," << cuda_ops << ","
            << achievable_mops(cuda_ops, kernel_stats.avg_ms) << ","
            << (all_correct ? "yes" : "no") << "\n";

        std::cout << std::setw(10) << n
                  << " | " << std::setw(14) << std::fixed << std::setprecision(3) << cpu_stats.avg_ms
                  << " | " << std::setw(14) << total_stats.avg_ms
                  << " | " << std::setw(12) << kernel_stats.avg_ms
                  << " | " << std::setw(8) << speedup
                  << " | " << std::setw(8) << (all_correct ? "yes" : "no") << std::endl;
    }

    csv.close();

    std::cout << "\nSaved results to task_b_cuda_results.csv" << std::endl;
    return EXIT_SUCCESS;
}


## 3. Compile the CUDA program


In [ ]:

!nvcc -O2 -std=c++17 task_b_cuda_merge_sort_colab.cu -o task_b_cuda_merge_sort_colab


## 4. Run the program

The arguments are:

```bash
./task_b_cuda_merge_sort_colab <cuda_block_size> <trials>
```

For the assignment, 256 threads per block and 5 trials is a normal choice.


In [ ]:

!./task_b_cuda_merge_sort_colab 256 5


## 5. View the CSV result table


In [ ]:

import pandas as pd

df = pd.read_csv("task_b_cuda_results.csv")
df


## 6. Plot comparison graphs


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "task_b_cuda_results.csv"
df = pd.read_csv(csv_path)

print(df)

pivot = df.pivot(index="n", columns="method", values="avg_total_ms")
ax = pivot.plot(kind="bar", figsize=(9, 5))
ax.set_title("Task B: CUDA Merge Sort vs CPU Sequential")
ax.set_xlabel("Number of elements")
ax.set_ylabel("Average execution time (ms)")
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("task_b_cuda_results.png", dpi=160)
plt.show()

cuda_df = df[df["method"] == "CUDA GPU"].copy()
plt.figure(figsize=(8, 5))
plt.plot(cuda_df["n"], cuda_df["speedup_vs_cpu"], marker="o")
plt.title("Task B: CUDA SpeedUp Compared to Sequential CPU")
plt.xlabel("Number of elements")
plt.ylabel("SpeedUp")
plt.xscale("log")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("task_b_cuda_speedup.png", dpi=160)
plt.show()


## 7. Download result files from Colab

After running all cells, you can download:

- `task_b_cuda_results.csv`
- `task_b_cuda_results.png`
- `task_b_cuda_speedup.png`

These can be inserted into the final report.


In [ ]:

from google.colab import files

files.download("task_b_cuda_results.csv")
files.download("task_b_cuda_results.png")
files.download("task_b_cuda_speedup.png")
